# Imports

In [ ]:
import pandas as pd

# Pre-processing

In [12]:
###################
# Original Dataset
###################

# input path to compiled dataset
input_path = r'../data/mnz_compiled.csv'

# output path for cleaned dataset
output_path = r'../data/mnz_cleaned.csv'


##### pre-process...
df = pd.read_csv(input_path)
df.drop(columns=df.columns[:3], inplace=True)
df.drop(columns=['lithology', 'mnz_description', 'original', 'U WT%', 'Th WT%', 'Pb WT%'], inplace=True)
df.rename(columns={'moecher':'texture', 'mnz chemistry-zone':'texture_chem', 'Age Ma':'age', 'Error 1sig':'error'}, inplace=True)


##### drop analyses >5% analytical error...
df['percent'] = df['error'] / df['age'] * 100
mask = df['percent'] < 5
df = df.loc[mask]
df.sort_values('error', inplace=True)
df.reset_index(inplace=True)


##### impute negative oxide values with 0.01...
# NOTE: can't be negative & allows ILR transformation later
oxide_cols = df.columns[8:]
df[oxide_cols] = df[oxide_cols].apply(pd.to_numeric, errors='coerce')
df[oxide_cols] = df[oxide_cols].clip(lower=0.01, upper=None)


##### oxide sum closure (force all to sum to 100)...
df[df.columns[8:]] = df[df.columns[8:]].apply(lambda col: col.map(lambda x: x/col.sum() * 100), axis=1)


##### feature engineering...
# combine light rare earth elements...
lree_cols = ['La2O3', 'Ce2O3', 'Pr2O3', 'Nd2O3', 'Sm2O3', 'Eu2O3']
df['lree'] = df[lree_cols].sum(axis=1)

# combine heavy rare earth elements (leave Y separate)...
hree_cols = ['Gd2O3', 'Tb2O3', 'Dy2O3', 'Ho2O3', 'Er2O3', 'Tm2O3', 'Yb2O3']
df['HREE'] = df[hree_cols].sum(axis=1)


##### indicator for garnet-/cordierite- vs. no-garnet/no cordierite lithologies...
# garnet
mask_grt = (df['sample'] == 'WA15-4') | (df['sample'] == 'WA15-5')
df['garnet'] = 1
df.loc[mask_grt, 'garnet'] = 0

# cordierite
mask_crd = (df['sample'] == 'WA15-4') | (df['sample'] == 'WA15-5') | (df['sample'] == 'WA14-002')
df['cordierite'] = 0
df.loc[mask_crd, 'cordierite'] = 1


##### save cleaned dataset...
df.to_csv(output_path, index=False)


df.head()

,index,sample,zone,mnz,texture,texture_chem,n,age,error,CaO,...,Er2O3,Tm2O3,Yb2O3,SO3,PbO,percent,lree,HREE,garnet,cordierite
0,8,EB14-001,CMZI,m28,overgrowth,Y depleted overgrowth,6.0,348.85,0.683097,0.986477,...,0.034439,0.069372,0.009740,0.018835,0.096939,0.195814,59.804756,1.359903,1,0
1,111,P14-003-2B,MDHSZ,m50,core,Th enriched core,5.0,307.14,0.789124,1.420269,...,0.009937,0.019969,0.009937,0.018184,0.132687,0.256926,57.520514,0.670830,1,0
2,176,P14-005A-5B,CBSZ,m20,core,Y depleted core,NaN,356.00,0.784962,0.819304,...,0.014718,0.009812,0.009812,0.020605,0.079477,0.220495,61.972776,1.588568,1,0
3,114,M1605-2-L3,MDHSZ,m4,rim,Y depleted rim,5.0,298.70,1.346540,0.837540,...,0.009748,0.009748,0.009748,0.016339,0.082968,0.450800,61.526147,1.376610,1,0
4,31,WA14-002,CMZI,m22,rim,Y depleted rim,5.0,353.24,1.408622,0.966597,...,0.009641,0.060215,0.009641,0.014777,0.085454,0.398772,60.699177,1.389823,1,1
